# Phần 1. Tiền xử lý dữ liệu

In [31]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler



In [32]:
file_path = '../data/Life Expectancy Data.csv'  
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()
from sklearn.impute import SimpleImputer

if df['Life expectancy'].isna().any():
    df['Life expectancy'] = df['Life expectancy'].fillna(df['Life expectancy'].mean())

numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
numeric_cols = numeric_cols.drop(['Life expectancy', 'Year'])

if len(numeric_cols) > 0:
    imputer = SimpleImputer(strategy='mean')
    df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

df = df.drop('Country', axis=1)
df = pd.get_dummies(df, columns=['Status'], drop_first=True)

In [33]:
labels_path = '../results/clustering_result/cluster_labels_with_classification.csv'
labels_df = pd.read_csv(labels_path)

print(f"Số mẫu dữ liệu sạch: {len(df)}")
print(f"Số mẫu nhãn lưu: {len(labels_df)}")

if len(df) != len(labels_df):
    raise ValueError("Số lượng mẫu không khớp! Kiểm tra lại quá trình xử lý missing values.")

y = labels_df['Classification_Label'].values
X = df.drop('Life expectancy', axis=1).values  

with open('../results/clustering_result/classification_thresholds.json', 'r', encoding='utf-8') as f:
    threshold_info = json.load(f)

label_names = threshold_info['label_names']
print(f"\nNhãn phân loại: {label_names}")
print(pd.Series(y).value_counts().sort_index())

Số mẫu dữ liệu sạch: 2938
Số mẫu nhãn lưu: 2938

Nhãn phân loại: ['Thấp', 'Trung bình', 'Cao']
0    970
1    979
2    989
Name: count, dtype: int64


In [35]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

Train: (2056, 20), Test: (882, 20)


In [39]:
y0 = y[y == 0]
y1 = y[y == 1]
y2 = y[y == 2]

In [40]:
print(y0.shape)
print(y1.shape)
print(y2.shape)

(970,)
(979,)
(989,)


# Phần 2. Giảm chiều